In [3]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import t
import pandas as pd
import time

In [4]:
def find_max_independent_set_heuristic(graph):
    best_independent_set = set()  #stores the largest independent set found
    vertices = list(graph.nodes())  #list of vertices in the graph
    #for recording the time
    start_time = time.time ()

    for initial_vertex in vertices:
        independent_set = set()  #independent set for the current iteration
        independent_set.add(initial_vertex)

        #add neighbors of the initial vertex to the independent set if feasible
        neighbors = list(graph.neighbors(initial_vertex))
        for neighbor in neighbors:
            if all(not graph.has_edge(neighbor, node) for node in independent_set):
                independent_set.add(neighbor)

        #update adjacency matrix by removing edges adjacent to nodes in the independent set and their neighbors
        temp_graph = graph.copy()
        nodes_to_remove = set(independent_set)
        for node in independent_set:
            if node in temp_graph:
                nodes_to_remove.update(temp_graph.neighbors(node))
        temp_graph.remove_nodes_from(nodes_to_remove)

        #repeat the improvement process
        while temp_graph.number_of_edges() > 0:
            #find the vertex with the maximum degree
            max_degree_vertex = max(temp_graph.degree, key=lambda x: x[1])[0]

            #add neighbors of the maximum degree vertex if feasible
            neighbors = list(temp_graph.neighbors(max_degree_vertex))
            for neighbor in neighbors:
                if all(not graph.has_edge(neighbor, node) for node in independent_set):
                    independent_set.add(neighbor)

            #update the adjacency matrix
            nodes_to_remove = set(independent_set)
            for node in independent_set:
                if node in temp_graph:
                    nodes_to_remove.update(temp_graph.neighbors(node))
            temp_graph.remove_nodes_from(nodes_to_remove)

        #try to add remaining vertices to the independent set
        for vertex in graph.nodes():
            if vertex not in independent_set and all(
                neighbor not in independent_set for neighbor in graph.neighbors(vertex)
            ):
                independent_set.add(vertex)

        #try shifting vertices to create feasible additions
        for vertex in vertices:
            if vertex not in independent_set:
                neighbors = list(graph.neighbors(vertex))
                for neighbor in neighbors:
                    if neighbor in independent_set and len(neighbors) == 1:
                        independent_set.remove(neighbor)
                        independent_set.add(vertex)
                        break

        #update the best independent set if the current one is larger
        if len(independent_set) > len(best_independent_set):
            best_independent_set = independent_set

    #calculating the total time taken by the algorithm
    end_time = time.time ()
    total_time = end_time - start_time

    return best_independent_set,total_time

In [ ]:
def test_find_max_independent_set_heuristic():
    tests = {
        "empty_graph": nx.Graph(),
        "single_node": nx.Graph(),
        "two_nodes_no_edge": nx.Graph(),
        "complete_graph_4": nx.complete_graph(4),
        "bipartite_graph": nx.complete_bipartite_graph(3, 4),
        "disconnected_components": nx.Graph([(1, 2), (3, 4)]),
        "cyclic_graph": nx.cycle_graph(5),
        "sparse_large_graph": nx.gnm_random_graph(1000, 50),
        "dense_large_graph": nx.gnm_random_graph(100, 4500),
    }

    tests["single_node"].add_node(1)  #adding a single node
    tests["two_nodes_no_edge"].add_nodes_from([1, 2])  #adding two nodes with no edges

    for test_name, graph in tests.items():
        print(f"Running test: {test_name}")
        independent_set, execution_time = find_max_independent_set_heuristic(graph)
        print(f"Independent Set: {independent_set}, Size: {len(independent_set)}, Time: {execution_time:.6f}s")

In [15]:
test_find_max_independent_set_heuristic()

Running test: empty_graph
Independent Set: set(), Size: 0, Time: 0.000000s
Running test: single_node
Independent Set: {1}, Size: 1, Time: 0.000000s
Running test: two_nodes_no_edge
Independent Set: {1, 2}, Size: 2, Time: 0.000000s
Running test: complete_graph_4
Independent Set: {0}, Size: 1, Time: 0.000000s
Running test: bipartite_graph
Independent Set: {3, 4, 5, 6}, Size: 4, Time: 0.001013s
Running test: disconnected_components
Independent Set: {2, 4}, Size: 2, Time: 0.000000s
Running test: cyclic_graph
Independent Set: {0, 3}, Size: 2, Time: 0.000000s
Running test: sparse_large_graph
Independent Set: {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 20, 21, 22, 23, 25, 26, 27, 28, 29, 30, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 103, 106, 107